# 04 · Optimización de hiperparámetros del detector YOLO11-s (Workshop S5)

**SIRAP – Proyecto Integrador en IA (MIAR0545)** · Eduardo Alejandro Ceballos Jijón · Christian Xavier Valle Maridueña

Este notebook estudia **cuánto y cómo** influye cada hiperparámetro del entrenamiento en el mAP del detector:

1. búsqueda bayesiana (TPE, Optuna) con corridas cortas y comparables;
2. **ranking de importancia** (fANOVA y permutación sobre un *random forest* sustituto);
3. **partial dependence plots** 1D y 2D (interacciones);
4. reentrenamiento de la mejor configuración con el protocolo completo y comparación **antes / después en *test***.

Todo se guarda en `MyDrive/proyecto_integrador_placas/optimizacion/`; si la sesión se cae, el estudio de Optuna
continúa desde el último *trial* (base SQLite en Drive). Requiere GPU (T4) y el Secret `ROBOFLOW_API_KEY`.
Documentación: `docs/optimizacion.md`.

In [ ]:
!pip install -q "ultralytics==8.4.177" "roboflow==1.7.1" "optuna>=4.0" scikit-learn

In [ ]:
import json, os, shutil
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import optuna
import pandas as pd
import torch
from ultralytics import YOLO

def get_secret(name):
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        return os.environ.get(name)

WORK_DIR = Path("/content/proyecto_integrador_placas")
try:
    from google.colab import drive
    drive.mount("/content/drive")
    WORK_DIR = Path("/content/drive/MyDrive/proyecto_integrador_placas")
except ImportError:
    pass
OPT_DIR = WORK_DIR / "optimizacion"
FIG_DIR = OPT_DIR / "figuras"
for d in (OPT_DIR, FIG_DIR):
    d.mkdir(parents=True, exist_ok=True)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no disponible")

## 1. Dataset
El mismo dataset y los mismos splits que el entrenamiento del modelo entregado (Roboflow v11), copiados al disco
local de Colab.

In [ ]:
RF_WORKSPACE, RF_PROJECT, RF_VERSION = "roboflow-universe-projects", "license-plate-recognition-rxg4e", 11
DATASET_DIR = Path("/content/datasets") / f"{RF_PROJECT}-v{RF_VERSION}"
if not (DATASET_DIR / "data.yaml").exists():
    from roboflow import Roboflow
    Roboflow(api_key=get_secret("ROBOFLOW_API_KEY")).workspace(RF_WORKSPACE).project(RF_PROJECT) \
        .version(RF_VERSION).download("yolov11", location=str(DATASET_DIR))
import yaml
cfg = yaml.safe_load((DATASET_DIR / "data.yaml").read_text())
for split, folder in (("train", "train"), ("val", "valid"), ("test", "test")):
    cfg[split] = str(DATASET_DIR / folder / "images")
cfg["path"] = str(DATASET_DIR)
DATA_YAML = DATASET_DIR / "data_abs.yaml"
DATA_YAML.write_text(yaml.safe_dump(cfg))
print(cfg["names"], "→", DATA_YAML)

## 2. Espacio de búsqueda y función objetivo

Corridas **cortas y comparables**: 10 épocas, 30 % de *train*, semilla 0. Se fija `optimizer="AdamW"`: con
`"auto"` Ultralytics ignora `lr0`.

In [ ]:
SEARCH_EPOCHS, SEARCH_FRACTION, N_TRIALS, SEED = 10, 0.30, 24, 0
BASE = {"lr0": 0.002, "weight_decay": 0.0005, "imgsz": 640, "batch": 16, "mosaic": 1.0, "hsv_v": 0.4, "cos_lr": False}

def suggest(trial):
    return {
        "lr0": trial.suggest_float("lr0", 1e-4, 1e-2, log=True),
        "weight_decay": trial.suggest_float("weight_decay", 1e-5, 1e-3, log=True),
        "imgsz": trial.suggest_categorical("imgsz", [512, 640, 768]),
        "batch": trial.suggest_categorical("batch", [8, 16, 32]),
        "mosaic": trial.suggest_float("mosaic", 0.0, 1.0),
        "hsv_v": trial.suggest_float("hsv_v", 0.0, 0.8),
        "cos_lr": trial.suggest_categorical("cos_lr", [False, True]),
    }

def train_short(params, name):
    model = YOLO("yolo11s.pt")
    model.train(data=str(DATA_YAML), epochs=SEARCH_EPOCHS, fraction=SEARCH_FRACTION, seed=SEED, deterministic=True,
                optimizer="AdamW", workers=2, project=str(OPT_DIR / "runs"), name=name, exist_ok=True,
                plots=False, verbose=False, **params)
    metrics = model.val(data=str(DATA_YAML), split="val", imgsz=params["imgsz"], verbose=False, plots=False)
    return float(metrics.box.map), float(metrics.box.map50)

def objective(trial):
    params = suggest(trial)
    map50_95, map50 = train_short(params, f"trial_{trial.number:03d}")
    trial.set_user_attr("map50", map50)
    torch.cuda.empty_cache()
    return map50_95

In [ ]:
study = optuna.create_study(
    study_name="sirap_yolo11s", direction="maximize", load_if_exists=True,
    storage=f"sqlite:///{OPT_DIR / 'optuna.db'}",
    sampler=optuna.samplers.TPESampler(seed=SEED, n_startup_trials=6))
if not any(t.params == BASE for t in study.trials):
    study.enqueue_trial(BASE)  # la configuración base también entra al estudio, para comparar en igualdad
remaining = N_TRIALS - len([t for t in study.trials if t.state.is_finished()])
if remaining > 0:
    study.optimize(objective, n_trials=remaining, gc_after_trial=True)
trials = study.trials_dataframe(attrs=("number", "value", "params", "user_attrs", "state"))
trials = trials[trials["state"] == "COMPLETE"].rename(columns=lambda c: c.replace("params_", "").replace("user_attrs_", ""))
trials = trials.rename(columns={"value": "map50_95"})
trials.to_csv(OPT_DIR / "optimizacion_trials.csv", index=False)
print("Mejor:", study.best_params, "mAP@0.5:0.95 =", round(study.best_value, 4))
trials.sort_values("map50_95", ascending=False).head(10)

## 3. Ranking de importancia
**fANOVA** (varianza explicada por cada hiperparámetro) y, para contrastar, **importancia por permutación** de un
*random forest* ajustado a los *trials*.

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.inspection import permutation_importance

PARAMS = ["lr0", "weight_decay", "imgsz", "batch", "mosaic", "hsv_v", "cos_lr"]
X = trials[PARAMS].copy()
X["lr0"], X["weight_decay"] = np.log10(X["lr0"]), np.log10(X["weight_decay"])
X["cos_lr"] = X["cos_lr"].astype(int)
y = trials["map50_95"].values

fanova = optuna.importance.get_param_importances(study, evaluator=optuna.importance.FanovaImportanceEvaluator(seed=SEED))
rf = RandomForestRegressor(n_estimators=500, min_samples_leaf=2, random_state=SEED).fit(X, y)
perm = permutation_importance(rf, X, y, n_repeats=30, random_state=SEED)
ranking = pd.DataFrame({"fanova": pd.Series(fanova), "permutacion": pd.Series(perm.importances_mean, index=PARAMS)}) \
    .sort_values("fanova", ascending=False)
ranking.to_csv(OPT_DIR / "optimizacion_importancia.csv")

fig, ax = plt.subplots(figsize=(8, 4))
ranking.plot.barh(ax=ax, color=["#2156e8", "#f7b900"]); ax.invert_yaxis()
ax.set_xlabel("Importancia"); ax.set_title("Importancia de hiperparámetros (mAP@0.5:0.95 en val)")
fig.tight_layout(); fig.savefig(FIG_DIR / "opt_importancia.png", dpi=160); plt.show()
ranking

## 4. Partial dependence plots
Efecto promedio de cada hiperparámetro sobre el mAP predicho por el sustituto (`lr0` y `weight_decay` en log10).

In [ ]:
from sklearn.inspection import PartialDependenceDisplay

fig, axes = plt.subplots(2, 4, figsize=(16, 7)); axes = axes.ravel()
PartialDependenceDisplay.from_estimator(rf, X, PARAMS, ax=axes[:len(PARAMS)], grid_resolution=30)
axes[-1].axis("off")
fig.suptitle("Partial dependence – mAP@0.5:0.95 (val)"); fig.tight_layout()
fig.savefig(FIG_DIR / "opt_pdp.png", dpi=160); plt.show()

### Interacciones
PDP 2D de los dos pares formados por los tres hiperparámetros más importantes.

In [ ]:
top = list(ranking.index[:3])
pairs = [(top[0], top[1]), (top[0], top[2])]
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
PartialDependenceDisplay.from_estimator(rf, X, pairs, ax=axes, grid_resolution=20)
fig.suptitle("Interacciones (PDP 2D)"); fig.tight_layout()
fig.savefig(FIG_DIR / "opt_interacciones.png", dpi=160); plt.show()
print("Pares:", pairs)

## 5. Antes / después con el protocolo completo
Se reentrena la mejor configuración con el protocolo del modelo entregado (40 épocas, 100 % de los datos,
paciencia 10) y se compara con la base **en *test***. La base se lee de los pesos ya entrenados (no se reentrena).

In [ ]:
BASE_WEIGHTS = WORK_DIR / "experimentos" / f"{RF_PROJECT}-v{RF_VERSION}" / "runs" / "YOLO11_yolo11s_completo" / "weights" / "best.pt"
best = dict(study.best_params)
final_dir = OPT_DIR / "runs" / "final_optimizado"
if not (final_dir / "weights" / "best.pt").exists():
    YOLO("yolo11s.pt").train(data=str(DATA_YAML), epochs=40, patience=10, seed=SEED, deterministic=True,
                             optimizer="AdamW", workers=2, project=str(OPT_DIR / "runs"), name="final_optimizado",
                             exist_ok=True, **best)

def test_metrics(weights, imgsz):
    m = YOLO(str(weights)).val(data=str(DATA_YAML), split="test", imgsz=imgsz, verbose=False, plots=False)
    return {"map50": float(m.box.map50), "map50_95": float(m.box.map), "precision": float(m.box.mp), "recall": float(m.box.mr)}

comparison = pd.DataFrame({
    "base": test_metrics(BASE_WEIGHTS, 640),
    "optimizada": test_metrics(final_dir / "weights" / "best.pt", best["imgsz"]),
}).T
comparison["delta_map50_95"] = comparison["map50_95"] - comparison.loc["base", "map50_95"]
comparison.to_csv(OPT_DIR / "optimizacion_antes_despues.csv")
json.dump({"mejor_config": best, "comparacion_test": comparison.to_dict()},
          open(OPT_DIR / "optimizacion_resumen.json", "w"), indent=2)
comparison

## 6. Copiar resultados al repositorio
Descargue estos archivos y súbalos a GitHub (o use `git` desde Colab):

| Archivo en Drive | Destino en el repositorio |
|---|---|
| `optimizacion/optimizacion_*.csv`, `optimizacion_resumen.json` | `results/metrics/` |
| `optimizacion/figuras/opt_*.png` | `results/figures/` |

Luego complete las tablas marcadas con ⏳ en `docs/optimizacion.md`.